<a href="https://colab.research.google.com/github/zhangling297/MAT_CS_599_deepLearningClassPracitices/blob/main/Copy_of_Mat599_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Downloads the NHANES files
## Builds the analytic dataset
### Produces the three comparison tables.

#### Setup
#1. The CDC P_DEMO file provides demographics plus the special 2017–March 2020 weights WTINTPRP and WTMECPRP
## The design variables SDMVSTRA and SDMVPSU; P_BMX contains BMXBMI; and P_DIQ contains DIQ010 for diagnosed diabetes.
### For 2017–March 2020 pre-pandemic NHANES analyses, CDC says to use the special pre-pandemic weights;
####This project merges questionnaire and examination data, WTMECPRP is the appropriate weight to use.

##### Project steps

# - Download P_DEMO.xpt, P_BMX.xpt, and P_DIQ.xpt.
#  - Read each XPT file into pandas.
# - Merge them by SEQN.
# - Keep adults aged 18+.

# - Create outcome indicators:

#  - obesity = 1 if BMXBMI >= 30

# - diabetes = 1 if DIQ010 == 1

# Create demographic groups:

# age groups, sex , race/ethnicity

# Use WTMECPRP to compute weighted prevalence.

# Build: Table 1 Demographic distribution comparison

# Table 2 Risk factors in clinical and person comparison

# Table 3 Summary comparison

In [ ]:
# ============================================================
# NHANES 2017-March 2020 class project:
# Comparative tables for diabetes vs obesity
#
# Files used:
#   P_DEMO.xpt : demographics + survey design + weights
#   P_BMX.xpt  : body measures (includes BMI)
#   P_DIQ.xpt  : diabetes questionnaire
#
# Outputs:
#   - table1_demographic_distribution.csv
#   - table2_risk_factors.csv
#   - table3_summary_comparison.csv
# ============================================================

import os
import requests
from io import BytesIO
import pandas as pd
import numpy as np

# -----------------------------
# 1. CDC file URLs
# -----------------------------
URLS = {
    "P_DEMO": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt",
    "P_BMX":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt",
    "P_DIQ":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.xpt",
}

DATA_DIR = "nhanes_data"
os.makedirs(DATA_DIR, exist_ok=True)

# -----------------------------
# 2. Downloader
# -----------------------------
def download_file(url: str, out_path: str) -> None:
    """Download file from CDC if not already present."""
    if os.path.exists(out_path):
        print(f"Already exists: {out_path}")
        return

    print(f"Downloading {url} ...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    print(f"Saved to {out_path}")

# Download all files
for name, url in URLS.items():
    download_file(url, os.path.join(DATA_DIR, f"{name}.xpt"))

# -----------------------------
# 3. Reader
# -----------------------------
def read_xpt(path: str) -> pd.DataFrame:
    """Read SAS transport (.xpt) file into pandas."""
    return pd.read_sas(path, format="xport")

demo = read_xpt(os.path.join(DATA_DIR, "P_DEMO.xpt"))
bmx  = read_xpt(os.path.join(DATA_DIR, "P_BMX.xpt"))
diq  = read_xpt(os.path.join(DATA_DIR, "P_DIQ.xpt"))

# -----------------------------
# 4. Keep needed columns only
# -----------------------------
demo_cols = [
    "SEQN",       # respondent ID
    "RIDAGEYR",   # age in years
    "RIAGENDR",   # sex
    "RIDRETH3",   # race/ethnicity
    "RIDSTATR",   # interview/exam status
    "WTMECPRP",   # MEC exam weight for 2017-Mar 2020
    "WTINTPRP",   # interview weight
    "SDMVSTRA",   # stratum
    "SDMVPSU"     # PSU
]
bmx_cols = [
    "SEQN",
    "BMXBMI",     # BMI
    "BMXWAIST"    # waist circumference
]
diq_cols = [
    "SEQN",
    "DIQ010"      # ever told by doctor has diabetes
]

demo = demo[demo_cols].copy()
bmx  = bmx[bmx_cols].copy()
diq  = diq[diq_cols].copy()

# -----------------------------
# 5. Merge datasets
# -----------------------------
df = demo.merge(bmx, on="SEQN", how="left").merge(diq, on="SEQN", how="left")

# -----------------------------
# 6. Restrict to adults
# -----------------------------
df = df[df["RIDAGEYR"] >= 18].copy()

# Optional:
# if you want to use only participants with MEC exam status:
# RIDSTATR == 2 means interviewed + examined
df = df[df["RIDSTATR"] == 2].copy()

# -----------------------------
# 7. Clean values / create outcomes
# -----------------------------
# Convert to numeric just in case
for col in ["RIDAGEYR", "RIAGENDR", "RIDRETH3", "WTMECPRP", "BMXBMI", "BMXWAIST", "DIQ010"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Outcome definitions
df["obesity"] = np.where(df["BMXBMI"] >= 30, 1, np.where(df["BMXBMI"].notna(), 0, np.nan))
df["diabetes"] = np.where(df["DIQ010"] == 1, 1, np.where(df["DIQ010"].isin([2, 3]), 0, np.nan))
# Note:
# NHANES DIQ010 coding commonly uses:
#   1 = Yes
#   2 = No
#   3 = Borderline
# For a simple class project, this script treats borderline as not diagnosed diabetes.
# You can instead exclude DIQ010 == 3 if your instructor prefers.

# Clinical / person-level risk indicators for comparison table
df["high_waist"] = np.where(
    ((df["RIAGENDR"] == 1) & (df["BMXWAIST"] > 102)) |
    ((df["RIAGENDR"] == 2) & (df["BMXWAIST"] > 88)),
    1,
    np.where(df["BMXWAIST"].notna() & df["RIAGENDR"].notna(), 0, np.nan)
)

# -----------------------------
# 8. Recode groups
# -----------------------------
def age_group(age):
    if pd.isna(age):
        return np.nan
    if 18 <= age <= 39:
        return "18-39"
    elif 40 <= age <= 59:
        return "40-59"
    else:
        return "60+"

sex_map = {
    1: "Male",
    2: "Female"
}

race_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other / Multiracial"
}

df["age_group"] = df["RIDAGEYR"].apply(age_group)
df["sex"] = df["RIAGENDR"].map(sex_map)
df["race_eth"] = df["RIDRETH3"].map(race_map)

# -----------------------------
# 9. Weighted helpers
# -----------------------------
WEIGHT = "WTMECPRP"

def weighted_prevalence(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    """Weighted prevalence (%) among non-missing observations."""
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return 100 * (temp[outcome] * temp[weight]).sum() / temp[weight].sum()

def weighted_mean(data: pd.DataFrame, var: str, weight: str = WEIGHT) -> float:
    """Weighted mean for continuous variables."""
    temp = data[[var, weight]].dropna()
    if temp.empty:
        return np.nan
    return np.average(temp[var], weights=temp[weight])

def weighted_numerator_estimate(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    """Weighted estimate of number with outcome (population estimate)."""
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return (temp[outcome] * temp[weight]).sum()

def weighted_denominator_estimate(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    """Weighted population denominator used in the prevalence estimate."""
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return temp[weight].sum()

# -----------------------------
# 10. Table 1
# Demographic distribution comparison
# -----------------------------
table1_rows = []

# Total
table1_rows.append({
    "Group": "Total adults",
    "Unweighted_n_diabetes": df["diabetes"].notna().sum(),
    "Unweighted_n_obesity": df["obesity"].notna().sum(),
    "Weighted_pop_for_diabetes": weighted_denominator_estimate(df, "diabetes"),
    "Weighted_pop_for_obesity": weighted_denominator_estimate(df, "obesity"),
    "Diabetes_prevalence_pct": weighted_prevalence(df, "diabetes"),
    "Obesity_prevalence_pct": weighted_prevalence(df, "obesity"),
})

# By age group
for g in ["18-39", "40-59", "60+"]:
    sub = df[df["age_group"] == g]
    table1_rows.append({
        "Group": f"Age {g}",
        "Unweighted_n_diabetes": sub["diabetes"].notna().sum(),
        "Unweighted_n_obesity": sub["obesity"].notna().sum(),
        "Weighted_pop_for_diabetes": weighted_denominator_estimate(sub, "diabetes"),
        "Weighted_pop_for_obesity": weighted_denominator_estimate(sub, "obesity"),
        "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
        "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
    })

# By sex
for g in ["Male", "Female"]:
    sub = df[df["sex"] == g]
    table1_rows.append({
        "Group": g,
        "Unweighted_n_diabetes": sub["diabetes"].notna().sum(),
        "Unweighted_n_obesity": sub["obesity"].notna().sum(),
        "Weighted_pop_for_diabetes": weighted_denominator_estimate(sub, "diabetes"),
        "Weighted_pop_for_obesity": weighted_denominator_estimate(sub, "obesity"),
        "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
        "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
    })

# By race/ethnicity
race_order = [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other / Multiracial"
]

for g in race_order:
    sub = df[df["race_eth"] == g]
    table1_rows.append({
        "Group": g,
        "Unweighted_n_diabetes": sub["diabetes"].notna().sum(),
        "Unweighted_n_obesity": sub["obesity"].notna().sum(),
        "Weighted_pop_for_diabetes": weighted_denominator_estimate(sub, "diabetes"),
        "Weighted_pop_for_obesity": weighted_denominator_estimate(sub, "obesity"),
        "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
        "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
    })

table1 = pd.DataFrame(table1_rows)

# -----------------------------
# 11. Table 2
# Risk factors in clinical and person comparison
# -----------------------------
table2_rows = []

# Diabetes group vs non-diabetes
for label, condition in [
    ("Has diabetes", df["diabetes"] == 1),
    ("No diabetes", df["diabetes"] == 0),
    ("Has obesity", df["obesity"] == 1),
    ("No obesity", df["obesity"] == 0),
]:
    sub = df[condition].copy()

    table2_rows.append({
        "Comparison_group": label,
        "Unweighted_n": len(sub),
        "Weighted_population": sub[WEIGHT].sum(skipna=True),
        "Mean_age_years": weighted_mean(sub, "RIDAGEYR"),
        "Mean_BMI": weighted_mean(sub, "BMXBMI"),
        "High_waist_pct": weighted_prevalence(sub, "high_waist"),
        "Female_pct": weighted_prevalence(
            sub.assign(is_female=np.where(sub["sex"] == "Female", 1, 0)),
            "is_female"
        ),
        "Age_60plus_pct": weighted_prevalence(
            sub.assign(age60=np.where(sub["RIDAGEYR"] >= 60, 1, 0)),
            "age60"
        ),
        "Obesity_pct_within_group": weighted_prevalence(sub, "obesity"),
        "Diabetes_pct_within_group": weighted_prevalence(sub, "diabetes"),
    })

table2 = pd.DataFrame(table2_rows)

# -----------------------------
# 12. Table 3
# Summary comparison
# -----------------------------
# For a simple class project, keep this table NHANES-based
# so one source is used consistently.
table3 = pd.DataFrame([
    {
        "Indicator": "Adult analytic sample (unweighted)",
        "Diabetes": int(df["diabetes"].notna().sum()),
        "Obesity": int(df["obesity"].notna().sum()),
    },
    {
        "Indicator": "Weighted adult population denominator",
        "Diabetes": weighted_denominator_estimate(df, "diabetes"),
        "Obesity": weighted_denominator_estimate(df, "obesity"),
    },
    {
        "Indicator": "Weighted prevalence (%)",
        "Diabetes": weighted_prevalence(df, "diabetes"),
        "Obesity": weighted_prevalence(df, "obesity"),
    },
    {
        "Indicator": "Weighted population estimate with condition",
        "Diabetes": weighted_numerator_estimate(df, "diabetes"),
        "Obesity": weighted_numerator_estimate(df, "obesity"),
    },
    {
        "Indicator": "Mean age (years)",
        "Diabetes": weighted_mean(df[df["diabetes"] == 1], "RIDAGEYR"),
        "Obesity": weighted_mean(df[df["obesity"] == 1], "RIDAGEYR"),
    },
    {
        "Indicator": "Mean BMI",
        "Diabetes": weighted_mean(df[df["diabetes"] == 1], "BMXBMI"),
        "Obesity": weighted_mean(df[df["obesity"] == 1], "BMXBMI"),
    },
    {
        "Indicator": "High waist circumference prevalence (%)",
        "Diabetes": weighted_prevalence(df[df["diabetes"] == 1], "high_waist"),
        "Obesity": weighted_prevalence(df[df["obesity"] == 1], "high_waist"),
    }
])

# -----------------------------
# 13. Round and save outputs
# -----------------------------
def round_numeric(df_in: pd.DataFrame, digits: int = 1) -> pd.DataFrame:
    df_out = df_in.copy()
    num_cols = df_out.select_dtypes(include=[np.number]).columns
    df_out[num_cols] = df_out[num_cols].round(digits)
    return df_out

table1_out = round_numeric(table1, 1)
table2_out = round_numeric(table2, 1)
table3_out = round_numeric(table3, 1)

table1_out.to_csv("table1_demographic_distribution.csv", index=False)
table2_out.to_csv("table2_risk_factors.csv", index=False)
table3_out.to_csv("table3_summary_comparison.csv", index=False)

print("\n=== Table 1. Demographic distribution comparison ===")
print(table1_out.to_string(index=False))

print("\n=== Table 2. Risk factors in clinical and person comparison ===")
print(table2_out.to_string(index=False))

print("\n=== Table 3. Summary comparison ===")
print(table3_out.to_string(index=False))

print("\nSaved:")
print(" - table1_demographic_distribution.csv")
print(" - table2_risk_factors.csv")
print(" - table3_summary_comparison.csv")

In [ ]:
# ============================================================
# GOOGLE COLAB FULL SCRIPT
# NHANES Diabetes and Obesity Comparison Project
#
# Tasks:
# 1. Download NHANES data and save 3 comparison tables
# 2. Create a copy of this script + README.md
# 3. Push files to your GitHub repository
# ============================================================

# =========================
# 0. Install packages
# =========================
!pip -q install pandas requests openpyxl

import os
import requests
from io import BytesIO
import pandas as pd
import numpy as np
import textwrap
import subprocess
from google.colab import files

# =========================
# 1. Project settings
# =========================
PROJECT_NAME = "nhanes-diabetes-obesity-project"
OUTPUT_DIR = os.path.join(PROJECT_NAME, "outputs")
DATA_DIR = os.path.join(PROJECT_NAME, "data")
SCRIPT_NAME = "nhanes_diabetes_obesity_project.py"
README_NAME = "README.md"

os.makedirs(PROJECT_NAME, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# =========================
# 2. NHANES file URLs
# =========================
URLS = {
    "P_DEMO": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt",
    "P_BMX":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt",
    "P_DIQ":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.xpt",
}

# =========================
# 3. Download NHANES files
# =========================
def download_file(url: str, out_path: str) -> None:
    if os.path.exists(out_path):
        print(f"Already exists: {out_path}")
        return
    print(f"Downloading: {url}")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    print(f"Saved: {out_path}")

for name, url in URLS.items():
    download_file(url, os.path.join(DATA_DIR, f"{name}.xpt"))

# =========================
# 4. Read XPT files
# =========================
def read_xpt(path: str) -> pd.DataFrame:
    return pd.read_sas(path, format="xport")

demo = read_xpt(os.path.join(DATA_DIR, "P_DEMO.xpt"))
bmx  = read_xpt(os.path.join(DATA_DIR, "P_BMX.xpt"))
diq  = read_xpt(os.path.join(DATA_DIR, "P_DIQ.xpt"))

print("Loaded files:")
print("DEMO:", demo.shape)
print("BMX :", bmx.shape)
print("DIQ :", diq.shape)

# =========================
# 5. Keep needed columns
# =========================
demo_cols = [
    "SEQN",
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "RIDSTATR",
    "WTMECPRP",
    "WTINTPRP",
    "SDMVSTRA",
    "SDMVPSU"
]

bmx_cols = [
    "SEQN",
    "BMXBMI",
    "BMXWAIST"
]

diq_cols = [
    "SEQN",
    "DIQ010"
]

demo = demo[demo_cols].copy()
bmx  = bmx[bmx_cols].copy()
diq  = diq[diq_cols].copy()

# =========================
# 6. Merge datasets
# =========================
df = demo.merge(bmx, on="SEQN", how="left").merge(diq, on="SEQN", how="left")
print("Merged shape:", df.shape)

# =========================
# 7. Restrict sample
# =========================
df = df[df["RIDAGEYR"] >= 18].copy()
df = df[df["RIDSTATR"] == 2].copy()   # interviewed + MEC examined
print("Adult MEC sample shape:", df.shape)

# =========================
# 8. Clean and define variables
# =========================
for col in ["RIDAGEYR", "RIAGENDR", "RIDRETH3", "WTMECPRP", "BMXBMI", "BMXWAIST", "DIQ010"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Outcomes
df["obesity"] = np.where(df["BMXBMI"] >= 30, 1,
                         np.where(df["BMXBMI"].notna(), 0, np.nan))

# Simple class-project rule:
# 1 = yes diabetes
# 2 = no
# 3 = borderline -> treated as 0 here
df["diabetes"] = np.where(df["DIQ010"] == 1, 1,
                          np.where(df["DIQ010"].isin([2, 3]), 0, np.nan))

# High waist circumference
df["high_waist"] = np.where(
    ((df["RIAGENDR"] == 1) & (df["BMXWAIST"] > 102)) |
    ((df["RIAGENDR"] == 2) & (df["BMXWAIST"] > 88)),
    1,
    np.where(df["BMXWAIST"].notna() & df["RIAGENDR"].notna(), 0, np.nan)
)

# Age groups
def age_group(age):
    if pd.isna(age):
        return np.nan
    if 18 <= age <= 39:
        return "18-39"
    elif 40 <= age <= 59:
        return "40-59"
    else:
        return "60+"

df["age_group"] = df["RIDAGEYR"].apply(age_group)

# Sex labels
sex_map = {1: "Male", 2: "Female"}
df["sex"] = df["RIAGENDR"].map(sex_map)

# Race/ethnicity labels
race_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other / Multiracial"
}
df["race_eth"] = df["RIDRETH3"].map(race_map)

# =========================
# 9. Weighted helper functions
# =========================
WEIGHT = "WTMECPRP"

def weighted_prevalence(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return 100 * (temp[outcome] * temp[weight]).sum() / temp[weight].sum()

def weighted_mean(data: pd.DataFrame, var: str, weight: str = WEIGHT) -> float:
    temp = data[[var, weight]].dropna()
    if temp.empty:
        return np.nan
    return np.average(temp[var], weights=temp[weight])

def weighted_denominator(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return temp[weight].sum()

def weighted_numerator(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return (temp[outcome] * temp[weight]).sum()

# =========================
# 10. Table 1
# Demographic distribution comparison
# =========================
table1_rows = []

def add_table1_row(label, sub):
    table1_rows.append({
        "Group": label,
        "Unweighted_n_diabetes": sub["diabetes"].notna().sum(),
        "Unweighted_n_obesity": sub["obesity"].notna().sum(),
        "Weighted_pop_diabetes": weighted_denominator(sub, "diabetes"),
        "Weighted_pop_obesity": weighted_denominator(sub, "obesity"),
        "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
        "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
    })

add_table1_row("Total adults", df)

for g in ["18-39", "40-59", "60+"]:
    add_table1_row(f"Age {g}", df[df["age_group"] == g])

for g in ["Male", "Female"]:
    add_table1_row(g, df[df["sex"] == g])

for g in [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other / Multiracial"
]:
    add_table1_row(g, df[df["race_eth"] == g])

table1 = pd.DataFrame(table1_rows)

# =========================
# 11. Table 2
# Risk factors in clinical and person comparison
# =========================
table2_rows = []

comparison_groups = [
    ("Has diabetes", df["diabetes"] == 1),
    ("No diabetes", df["diabetes"] == 0),
    ("Has obesity", df["obesity"] == 1),
    ("No obesity", df["obesity"] == 0),
]

for label, mask in comparison_groups:
    sub = df[mask].copy()
    sub["is_female"] = np.where(sub["sex"] == "Female", 1, 0)
    sub["age60plus"] = np.where(sub["RIDAGEYR"] >= 60, 1, 0)

    table2_rows.append({
        "Comparison_group": label,
        "Unweighted_n": len(sub),
        "Weighted_population": sub[WEIGHT].sum(skipna=True),
        "Mean_age_years": weighted_mean(sub, "RIDAGEYR"),
        "Mean_BMI": weighted_mean(sub, "BMXBMI"),
        "High_waist_pct": weighted_prevalence(sub, "high_waist"),
        "Female_pct": weighted_prevalence(sub, "is_female"),
        "Age_60plus_pct": weighted_prevalence(sub, "age60plus"),
        "Obesity_pct_within_group": weighted_prevalence(sub, "obesity"),
        "Diabetes_pct_within_group": weighted_prevalence(sub, "diabetes"),
    })

table2 = pd.DataFrame(table2_rows)

# =========================
# 12. Table 3
# Summary comparison
# =========================
table3 = pd.DataFrame([
    {
        "Indicator": "Adult analytic sample (unweighted)",
        "Diabetes": df["diabetes"].notna().sum(),
        "Obesity": df["obesity"].notna().sum(),
    },
    {
        "Indicator": "Weighted adult population denominator",
        "Diabetes": weighted_denominator(df, "diabetes"),
        "Obesity": weighted_denominator(df, "obesity"),
    },
    {
        "Indicator": "Weighted prevalence (%)",
        "Diabetes": weighted_prevalence(df, "diabetes"),
        "Obesity": weighted_prevalence(df, "obesity"),
    },
    {
        "Indicator": "Weighted population estimate with condition",
        "Diabetes": weighted_numerator(df, "diabetes"),
        "Obesity": weighted_numerator(df, "obesity"),
    },
    {
        "Indicator": "Mean age (years) among cases",
        "Diabetes": weighted_mean(df[df["diabetes"] == 1], "RIDAGEYR"),
        "Obesity": weighted_mean(df[df["obesity"] == 1], "RIDAGEYR"),
    },
    {
        "Indicator": "Mean BMI among cases",
        "Diabetes": weighted_mean(df[df["diabetes"] == 1], "BMXBMI"),
        "Obesity": weighted_mean(df[df["obesity"] == 1], "BMXBMI"),
    },
    {
        "Indicator": "High waist circumference prevalence (%) among cases",
        "Diabetes": weighted_prevalence(df[df["diabetes"] == 1], "high_waist"),
        "Obesity": weighted_prevalence(df[df["obesity"] == 1], "high_waist"),
    },
])

# =========================
# 13. Round outputs
# =========================
def round_numeric(df_in, digits=1):
    out = df_in.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out

table1_out = round_numeric(table1, 1)
table2_out = round_numeric(table2, 1)
table3_out = round_numeric(table3, 1)

# =========================
# 14. Save tables locally in Colab
# Task 1
# =========================
table1_csv = os.path.join(OUTPUT_DIR, "table1_demographic_distribution.csv")
table2_csv = os.path.join(OUTPUT_DIR, "table2_risk_factors.csv")
table3_csv = os.path.join(OUTPUT_DIR, "table3_summary_comparison.csv")
analytic_csv = os.path.join(OUTPUT_DIR, "analytic_dataset.csv")
excel_path = os.path.join(OUTPUT_DIR, "nhanes_diabetes_obesity_tables.xlsx")

table1_out.to_csv(table1_csv, index=False)
table2_out.to_csv(table2_csv, index=False)
table3_out.to_csv(table3_csv, index=False)
df.to_csv(analytic_csv, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    table1_out.to_excel(writer, sheet_name="Table1_Demographic", index=False)
    table2_out.to_excel(writer, sheet_name="Table2_RiskFactors", index=False)
    table3_out.to_excel(writer, sheet_name="Table3_Summary", index=False)

print("\nSaved local files:")
print(table1_csv)
print(table2_csv)
print(table3_csv)
print(analytic_csv)
print(excel_path)

# =========================
# 15. Create README.md
# =========================
readme_text = f"""# NHANES Diabetes and Obesity Comparison Project

## Project Overview
This Google Colab project compares diabetes and obesity in U.S. adults using NHANES 2017-March 2020 pre-pandemic data.

## Research Tasks
1. Save the comparison tables used in the class project
2. Create a reusable script and README for GitHub

## Data Files Used
- `P_DEMO.xpt` for demographics and survey weights
- `P_BMX.xpt` for BMI and waist circumference
- `P_DIQ.xpt` for diagnosed diabetes questionnaire data

## Variables Used
### Demographic variables
- `RIDAGEYR`: age in years
- `RIAGENDR`: sex
- `RIDRETH3`: race/ethnicity

### Survey design variables
- `WTMECPRP`: MEC exam weight
- `WTINTPRP`: interview weight
- `SDMVSTRA`: stratum
- `SDMVPSU`: PSU

### Clinical variables
- `BMXBMI`: body mass index
- `BMXWAIST`: waist circumference
- `DIQ010`: doctor told you have diabetes

## Analytic Definitions
- Adults only: `RIDAGEYR >= 18`
- MEC examined sample: `RIDSTATR == 2`
- Obesity: `BMXBMI >= 30`
- Diabetes: `DIQ010 == 1`
- Weight used for prevalence estimates: `WTMECPRP`

## Outputs
- `outputs/table1_demographic_distribution.csv`
- `outputs/table2_risk_factors.csv`
- `outputs/table3_summary_comparison.csv`
- `outputs/analytic_dataset.csv`
- `outputs/nhanes_diabetes_obesity_tables.xlsx`

## Tables
### Table 1
Demographic distribution comparison of diabetes and obesity by:
- total adults
- age group
- sex
- race/ethnicity

### Table 2
Risk factors in clinical and person comparison:
- mean age
- mean BMI
- high waist circumference
- female percentage
- age 60+ percentage

### Table 3
Summary comparison:
- weighted prevalence
- weighted population estimate
- mean age
- mean BMI
- high waist prevalence

## Installation
This project is designed for Google Colab, but can also run in Python with:
```bash
pip install pandas requests openpyxl
```
"""